In [ ]:
# CELL 1 — Setup and validate uploaded dataset ZIP

from pathlib import Path
import zipfile
import pandas as pd

ZIP_PATH = Path("/content/indic_langid_hf.zip")
EXTRACT_DIR = Path("/content/dataset_extracted")  #  Fixed: Separate directory for extraction

if not ZIP_PATH.is_file() or not zipfile.is_zipfile(ZIP_PATH):
    raise ValueError("The uploaded file does not exist or is not a valid ZIP archive.")

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    bad_file = z.testzip()
    if bad_file:
        raise ValueError(f"Corrupt file inside ZIP: {bad_file}")

    # Extract safely into the target directory (preventing Path Traversal vulnerabilities)
    resolved_extract_dir = EXTRACT_DIR.resolve()
    for member in z.infolist():
        target = (resolved_extract_dir / member.filename).resolve()
        if resolved_extract_dir not in target.parents:
            raise ValueError(f"Unsafe ZIP path detected: {member.filename}")

    z.extractall(resolved_extract_dir)

# Locate the CSVs even if the ZIP contains a nested folder.
def find_file(filename):
    matches = list(EXTRACT_DIR.rglob(filename))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one '{filename}'; found {len(matches)}"
        )
    return matches[0]

TRAIN_PATH = find_file("train.csv")
VAL_PATH = find_file("validation.csv")
TEST_PATH = find_file("test.csv")

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

LABELS = [
    "as-IN", "bn-IN", "en-IN", "gu-IN", "hi-IN", "kn-IN",
    "ml-IN", "mr-IN", "pa-IN", "ta-IN", "te-IN", "none"
]

for name, df in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df)
]:
    required = {"text", "label"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{name}.csv is missing columns: {missing}")

    # if df["text"].isna().any() or df["label"].isna().any():
    #     raise ValueError(f"{name}.csv contains missing text or labels.")

    unknown = set(df["label"].astype(str).unique()) - set(LABELS)
    if unknown:
        raise ValueError(f"{name}.csv has unexpected labels: {unknown}")

    print(f"\n{name.upper()}: {len(df):,} rows")
    print(df["label"].value_counts().reindex(LABELS, fill_value=0))

print("\nAll three CSVs loaded and label checks passed.")
print("Train:", TRAIN_PATH)
print("Validation:", VAL_PATH)
print("Test:", TEST_PATH)

print("\nNext: train one shared multilingual tokenizer on TRAIN text only,")
print("then adapt and fine-tune E2 and E4 using the same tokenizer.")



TRAIN: 723,479 rows
label
as-IN    60822
bn-IN    60308
en-IN    60603
gu-IN    60956
hi-IN    60630
kn-IN    61002
ml-IN    60888
mr-IN    60858
pa-IN    60995
ta-IN    61161
te-IN    60914
none     54342
Name: count, dtype: int64

VALIDATION: 90,219 rows
label
as-IN    7583
bn-IN    7734
en-IN    7689
gu-IN    7397
hi-IN    7659
kn-IN    7538
ml-IN    7658
mr-IN    7521
pa-IN    7465
ta-IN    7428
te-IN    7639
none     6908
Name: count, dtype: int64

TEST: 90,459 rows
label
as-IN    7595
bn-IN    7958
en-IN    7708
gu-IN    7647
hi-IN    7711
kn-IN    7460
ml-IN    7454
mr-IN    7621
pa-IN    7540
ta-IN    7411
te-IN    7447
none     6907
Name: count, dtype: int64

All three CSVs loaded and label checks passed.
Train: /content/dataset_extracted/train.csv
Validation: /content/dataset_extracted/validation.csv
Test: /content/dataset_extracted/test.csv

Next: train one shared multilingual tokenizer on TRAIN text only,
then adapt and fine-tune E2 and E4 using the same tokenizer.


In [ ]:
import pandas as pd

In [ ]:
df=pd.read_csv("/content/dataset_extracted/train.csv")

In [ ]:
df.columns

Index(['text', 'label', 'label_id', 'category', 'script', 'source_type',
       'language_detail'],
      dtype='object')

In [ ]:
df.label.unique()

array(['pa-IN', 'en-IN', 'kn-IN', 'bn-IN', 'gu-IN', 'none', 'as-IN',
       'ml-IN', 'ta-IN', 'te-IN', 'mr-IN', 'hi-IN'], dtype=object)

In [ ]:
df[df.label=='hi-IN'].category.unique()

array(['hi_stt_items', 'hindi_only', 'hindi_english_mixed_script',
       'hi_stt_codemix', 'hi_romanized_auto', 'hi_romanized',
       'hi_stt_numsent', 'hi_stt_entitysent', 'hi_backchannel',
       'hi_stt_namesent', 'hi_stt_entitysent2', 'hi_restored',
       'hi_stt_short2', 'hi_stt_short'], dtype=object)

In [ ]:
df[df.label=='hi-IN'][df.category=='hi_stt_short']

/tmp/ipykernel_572/163725973.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df[df.label=='hi-IN'][df.category=='hi_stt_short']


,text,label,label_id,category,script,source_type,language_detail
268679,ठीक है thanks ठीक रहेगा,hi-IN,0,hi_stt_short,Devanagari,synthetic,hi-IN
403984,हाँ please बताइए एक बार,hi-IN,0,hi_stt_short,Devanagari,synthetic,hi-IN
426154,नहीं thanks ना,hi-IN,0,hi_stt_short,Devanagari,synthetic,hi-IN
495527,हाँ हाँ okay फिर से,hi-IN,0,hi_stt_short,Devanagari,synthetic,hi-IN
716000,नहीं थैंक्स ना.,hi-IN,0,hi_stt_short,Devanagari,synthetic,hi-IN


In [ ]:
# FIXED CELL 2 (v2) — multilingual cased WordPiece tokenizer
!pip -q install -U tokenizers transformers

from pathlib import Path
import json
import numpy as np
import pandas as pd

from tokenizers import Tokenizer, models, trainers, normalizers, pre_tokenizers, decoders, processors
from transformers import PreTrainedTokenizerFast

TOKENIZER_DIR = Path("/content/multilingual_wordpiece_tokenizer")
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)

VOCAB_SIZE = 30522
MIN_FREQUENCY = 2
MAX_LENGTH = 128

assert "train_df" in globals(), "Run the dataset loading cell first."
assert {"text", "label"} <= set(train_df.columns), "train_df needs text and label columns."

# ---- training text: train split only; keep numeric-id rows small so they don't eat the vocab
df_tok = train_df.dropna(subset=["text"]).copy()
df_tok["text"] = df_tok["text"].astype(str)
if "category" in df_tok.columns:
    is_num = df_tok["category"].eq("none_numeric_id")
    df_tok = pd.concat([df_tok[~is_num],
                        df_tok[is_num].sample(min(3000, int(is_num.sum())), random_state=42)])
training_texts = [t for t in df_tok["text"].tolist() if t.strip()]
assert len(training_texts) > 0, "No training text found."

def text_iterator():
    for t in training_texts:
        yield t            # normalization happens inside the tokenizer (same at train + inference)

special_tokens = ["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]

# characters that must never become [UNK]: ASCII, punctuation, rupee sign, native digits of every Indian script
initial_alphabet = list("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
                        ".,;:!?-_/\\@#%&*()[]{}'\"+=<>|~`^₹")
for base in (0x966, 0x9E6, 0xA66, 0xAE6, 0xB66, 0xBE6, 0xC66, 0xCE6, 0xD66):
    initial_alphabet += [chr(base + i) for i in range(10)]

backend = Tokenizer(models.WordPiece(unk_token="[UNK]"))
backend.normalizer = normalizers.Sequence([
    normalizers.NFKC(),
    normalizers.BertNormalizer(clean_text=True, handle_chinese_chars=True,
                               strip_accents=False, lowercase=False),
])
backend.pre_tokenizer = pre_tokenizers.BertPreTokenizer()
backend.decoder = decoders.WordPiece(prefix="##")

trainer = trainers.WordPieceTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQUENCY,
    special_tokens=special_tokens,
    limit_alphabet=6000,
    initial_alphabet=initial_alphabet,
    continuing_subword_prefix="##",
    show_progress=True,
)

print(f"Training on {len(training_texts):,} training sentences...")
backend.train_from_iterator(text_iterator(), trainer=trainer, length=len(training_texts))

actual_vocab_size = backend.get_vocab_size()
print("Actual vocabulary size:", actual_vocab_size)
if actual_vocab_size < 1000:
    raise RuntimeError(f"Tokenizer vocabulary is only {actual_vocab_size}. Training failed; do not fine-tune.")

# ---- add [CLS] ... [SEP] (the original cell never added them)
cls_id, sep_id = backend.token_to_id("[CLS]"), backend.token_to_id("[SEP]")
backend.post_processor = processors.TemplateProcessing(
    single="[CLS] $A [SEP]",
    pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[("[CLS]", cls_id), ("[SEP]", sep_id)],
)

backend.save(str(TOKENIZER_DIR / "tokenizer.json"))
backend.model.save(str(TOKENIZER_DIR))

hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=backend,
    unk_token="[UNK]", pad_token="[PAD]", cls_token="[CLS]", sep_token="[SEP]", mask_token="[MASK]",
    model_max_length=MAX_LENGTH,
)
hf_tokenizer.save_pretrained(str(TOKENIZER_DIR))

# ---- hard checks
ids = hf_tokenizer("Okay")["input_ids"]
assert ids[0] == hf_tokenizer.cls_token_id and ids[-1] == hf_tokenizer.sep_token_id, "CLS/SEP missing!"
assert hf_tokenizer.tokenize("Okay") != hf_tokenizer.tokenize("okay"), "Tokenizer is lower-casing!"
assert hf_tokenizer.unk_token_id not in hf_tokenizer("9876543210 +91 ₹500 ९८७६५४३२१०")["input_ids"], "digits became [UNK]"

samples = [
    "Eligibility क्या है?", "मुझे admission process बताओ", "What is the course fee?",
    "ਪੰਜਾਬੀ ਅਤੇ हिंदी दोनों", "मराठी आणि English mix", "తెలుగు మరియు English", "தமிழ் and English",
    "বাংলা and English", "ગુજરાતી and English", "ಕನ್ನಡ and English", "മലയാളം and English", "অসমীয়া and English",
    "mujhe refund chahiye abhi", "haan ji boliye", "okay", "Okay", "OKAY",
    "میرا نام احمد ہے", "मेरो नाम राम हो", "ମୋର ନାମ ରାମ", "9876543210", "MH12AB1234",
]
print("\nMULTILINGUAL TOKENIZATION CHECK")
for text in samples:
    print(f"\n{text}\n{hf_tokenizer.tokenize(text)}")

# ---- audit: global + per label (a weak language hides in a global average)
def audit_texts(texts):
    unk = tot = words = 0; lens = []
    for t in texts:
        enc = hf_tokenizer(t, add_special_tokens=False, truncation=False)["input_ids"]
        unk += sum(i == hf_tokenizer.unk_token_id for i in enc); tot += len(enc)
        words += max(len(t.split()), 1); lens.append(len(enc) + 2)
    return unk, tot, words, lens

rng = np.random.default_rng(42)
sample_size = min(10000, len(training_texts))
idx = rng.choice(len(training_texts), sample_size, replace=False)
unk, tot, words, lengths = audit_texts([training_texts[int(i)] for i in idx])
unk_rate = 100 * unk / max(tot, 1)

print("\nTOKENIZER AUDIT")
print("Vocabulary size:", len(hf_tokenizer))
print(f"UNK token rate: {unk_rate:.4f}%   tokens/word: {tot/words:.2f}")
print("Length P50/P90/P95/P99:", [int(np.percentile(lengths, p)) for p in (50, 90, 95, 99)])
print(f"Over {MAX_LENGTH} tokens: {100*np.mean(np.array(lengths) > MAX_LENGTH):.2f}%")

print("\nPER LABEL (UNK %, tokens per word)")
per_label = {}
for lab, g in df_tok.groupby("label"):
    s = g["text"].sample(min(1500, len(g)), random_state=1).tolist()
    u, t, w, _ = audit_texts(s)
    per_label[lab] = {"unk_percent": 100 * u / max(t, 1), "tokens_per_word": t / max(w, 1)}
    print(f"{lab:8s} UNK {per_label[lab]['unk_percent']:.3f}%   tok/word {per_label[lab]['tokens_per_word']:.2f}")

audit = {"vocab_size": len(hf_tokenizer), "training_rows": len(training_texts), "sample_size": sample_size,
         "unk_rate_percent": unk_rate, "mean_sequence_length": float(np.mean(lengths)),
         "p50": float(np.percentile(lengths, 50)), "p90": float(np.percentile(lengths, 90)),
         "p95": float(np.percentile(lengths, 95)), "p99": float(np.percentile(lengths, 99)),
         "max_length": MAX_LENGTH, "per_label": per_label}
with open(TOKENIZER_DIR / "tokenizer_audit.json", "w", encoding="utf-8") as f:
    json.dump(audit, f, indent=2)

print("\nTokenizer saved to:", TOKENIZER_DIR)
print("Validation passed.")

Training on 709,814 training sentences...
Actual vocabulary size: 30522

MULTILINGUAL TOKENIZATION CHECK

Eligibility क्या है?
['El', '##ig', '##ibility', 'क्या', 'है', '?']

मुझे admission process बताओ
['मुझे', 'admission', 'process', 'बता', '##ओ']

What is the course fee?
['What', 'is', 'the', 'course', 'fee', '?']

ਪੰਜਾਬੀ ਅਤੇ हिंदी दोनों
['ਪੰ', '##ਜਾ', '##ਬੀ', 'ਅਤੇ', 'हि', '##ंद', '##ी', 'दोनों']

मराठी आणि English mix
['म', '##रा', '##ठी', 'आणि', 'English', 'mix']

తెలుగు మరియు English
['తెలు', '##గు', 'మరియు', 'English']

தமிழ் and English
['தமிழ்', 'and', 'English']

বাংলা and English
['বা', '##ং', '##লা', 'and', 'English']

ગુજરાતી and English
['ગુજ', '##રાત', '##ી', 'and', 'English']

ಕನ್ನಡ and English
['ಕನ', '##್', '##ನ', '##ಡ', 'and', 'English']

മലയാളം and English
['മല', '##യ', '##ാള', '##ം', 'and', 'English']

অসমীয়া and English
['অসম', '##ীয়া', 'and', 'English']

mujhe refund chahiye abhi
['mujhe', 'refund', 'cha', '##hiy', '##e', 'abhi']

haan ji boliye
['haan', 'ji', '

In [ ]:

# ---------------- FIX DATAFRAME COLUMNS ----------------

for name, frame in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df)
]:
    # Your dataset uses "label", while the training code expects "final_label".
    if "final_label" not in frame.columns and "label" in frame.columns:
        frame.rename(columns={"label": "final_label"}, inplace=True)

    required = {"text", "final_label"}
    missing = required - set(frame.columns)

    if missing:
        raise ValueError(
            f"{name} dataframe is missing columns: {missing}. "
            f"Available columns: {list(frame.columns)}"
        )

print("Train columns:", train_df.columns.tolist())
print("Validation columns:", val_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())

print("\nTraining label distribution:")
print(train_df["final_label"].value_counts())


Train columns: ['text', 'final_label', 'label_id', 'category', 'script', 'source_type', 'language_detail']
Validation columns: ['text', 'final_label', 'label_id', 'category', 'script', 'source_type', 'language_detail']
Test columns: ['text', 'final_label', 'label_id', 'category', 'script', 'source_type', 'language_detail']

Training label distribution:
final_label
ta-IN    61161
kn-IN    61002
pa-IN    60995
gu-IN    60956
te-IN    60914
ml-IN    60888
mr-IN    60858
as-IN    60822
hi-IN    60630
en-IN    60603
bn-IN    60308
none     54342
Name: count, dtype: int64


In [ ]:
# CELL 3 — Fine-tune BERT-Tiny E2
# Requires: train_df, val_df, test_df from Cell 1
#           multilingual tokenizer from Cell 2
#
# DataFrame columns expected:
#   'text', 'label', 'label_id', 'category', 'script',
#   'source_type', 'language_detail'

!pip -q install -U transformers scikit-learn tqdm

import os
import json
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoTokenizer as OriginalTokenizer,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from tqdm.auto import tqdm

# ---------------- CONFIG ----------------

SEED = 42
CHECKPOINT = "google/bert_uncased_L-2_H-128_A-2"
TOKENIZER_DIR = "/content/multilingual_wordpiece_tokenizer"
OUTPUT_DIR = Path("/content/bert_tiny_e2_multilingual_version4")

LABELS = [
    "as-IN", "bn-IN", "en-IN", "gu-IN", "hi-IN", "kn-IN",
    "ml-IN", "mr-IN", "pa-IN", "ta-IN", "te-IN", "none"
]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

# Column names in your dataframes
TEXT_COL = "text"
LABEL_COL = "label"   # language label column (was "final_label")

MAX_LENGTH = 128
BATCH_SIZE = 64
EPOCHS = 3
ENCODER_LR = 5e-5
EMBEDDING_LR = 3e-4
CLASSIFIER_LR = 3e-4
WEIGHT_DECAY = 0.01

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not detected. In Colab, select Runtime > Change runtime type > GPU."
    )

device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)

# ---------------- VALIDATE DATAFRAMES ----------------

for name, frame in [
    ("train", train_df), ("validation", val_df), ("test", test_df)
]:
    if not {TEXT_COL, LABEL_COL}.issubset(frame.columns):
        raise ValueError(
            f"{name} dataframe is missing required columns. "
            f"Found: {list(frame.columns)}"
        )

    unexpected = set(frame[LABEL_COL].astype(str).unique()) - set(LABELS)
    if unexpected:
        raise ValueError(
            f"Unexpected labels in {name} dataframe: {sorted(unexpected)}\n"
            f"Expected only: {LABELS}\n"
            f"Tip: inspect with train_df['{LABEL_COL}'].value_counts() and "
            f"map them to the LABELS strings."
        )

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ---------------- LOAD TOKENIZERS ----------------

tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_DIR,
    use_fast=True
)

# Original tokenizer is used ONLY to transfer matching pretrained embeddings.
original_tokenizer = OriginalTokenizer.from_pretrained(CHECKPOINT)

assert len(tokenizer) == 30522, (
    f"Expected vocabulary size 32000, got {len(tokenizer)}"
)

# ---------------- LOAD PRETRAINED E2 ----------------

model = AutoModelForSequenceClassification.from_pretrained(
    CHECKPOINT,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

assert model.config.num_hidden_layers == 2
assert model.config.hidden_size == 128
assert model.config.num_attention_heads == 2

# ---------------- ADAPT TOKEN EMBEDDINGS ----------------
# New token IDs do not align with original English vocabulary IDs.
# Initialize all rows, then transfer embeddings for matching tokens.
# For Latin tokens, try lowercase matching as well.
# Unmatched Indic tokens receive random initialization.

old_vocab = original_tokenizer.get_vocab()
new_vocab = tokenizer.get_vocab()

old_embedding = (
    model.bert.embeddings.word_embeddings.weight.detach().clone()
)
new_embedding = torch.empty_like(old_embedding)

nn.init.normal_(
    new_embedding,
    mean=0.0,
    std=model.config.initializer_range
)

matched = 0
exact_matches = 0
lowercase_matches = 0

for token, new_id in new_vocab.items():
    old_id = old_vocab.get(token)

    if old_id is not None:
        new_embedding[new_id] = old_embedding[old_id]
        matched += 1
        exact_matches += 1
        continue

    lower_token = token.lower()
    old_id = old_vocab.get(lower_token)

    if old_id is not None:
        new_embedding[new_id] = old_embedding[old_id]
        matched += 1
        lowercase_matches += 1

# Keep padding embedding zero.
if tokenizer.pad_token_id is not None:
    new_embedding[tokenizer.pad_token_id].zero_()

with torch.no_grad():
    model.bert.embeddings.word_embeddings.weight.copy_(new_embedding)

print("\nEmbedding adaptation:")
print("New vocabulary:", len(new_vocab))
print("Exact pretrained token matches:", exact_matches)
print("Lowercase pretrained token matches:", lowercase_matches)
print("Total transferred rows:", matched)
print("Randomly initialized rows:", len(new_vocab) - matched)
print("Transfer coverage:", round(100 * matched / len(new_vocab), 2), "%")

# ---------------- DATASET ----------------

class TextDataset(Dataset):
    def __init__(self, frame):
        self.texts = frame[TEXT_COL].astype(str).tolist()
        self.labels = [
            LABEL2ID[x] for x in frame[LABEL_COL].astype(str).tolist()
        ]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_batch(batch):
    texts, labels = zip(*batch)
    encoded = tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )
    encoded["labels"] = torch.tensor(labels, dtype=torch.long)
    return encoded

train_loader = DataLoader(
    TextDataset(train_df),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    TextDataset(val_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    TextDataset(test_df),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=0,
    pin_memory=True
)

# ---------------- OPTIMIZER ----------------

embedding_params = []
classifier_params = []
encoder_params = []

for name, param in model.named_parameters():
    if "embeddings.word_embeddings" in name:
        embedding_params.append(param)
    elif name.startswith("classifier."):
        classifier_params.append(param)
    else:
        encoder_params.append(param)

optimizer = torch.optim.AdamW(
    [
        {
            "params": encoder_params,
            "lr": ENCODER_LR,
            "weight_decay": WEIGHT_DECAY
        },
        {
            "params": embedding_params,
            "lr": EMBEDDING_LR,
            "weight_decay": WEIGHT_DECAY
        },
        {
            "params": classifier_params,
            "lr": CLASSIFIER_LR,
            "weight_decay": WEIGHT_DECAY
        }
    ]
)

total_steps = len(train_loader) * EPOCHS
warmup_steps = int(0.05 * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

scaler = torch.cuda.amp.GradScaler(enabled=True)
model.to(device)

# ---------------- EVALUATION FUNCTION ----------------

def evaluate(loader):
    model.eval()
    all_preds = []
    all_targets = []
    losses = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating", leave=False):
            batch = {
                key: value.to(device, non_blocking=True)
                for key, value in batch.items()
            }

            with torch.cuda.amp.autocast(enabled=True):
                outputs = model(**batch)

            losses.append(outputs.loss.item())
            preds = outputs.logits.argmax(dim=-1)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_targets.extend(batch["labels"].cpu().numpy().tolist())

    return {
        "loss": float(np.mean(losses)),
        "accuracy": float(accuracy_score(all_targets, all_preds)),
        "macro_f1": float(
            f1_score(
                all_targets,
                all_preds,
                labels=list(range(len(LABELS))),
                average="macro",
                zero_division=0
            )
        ),
        "predictions": all_preds,
        "targets": all_targets
    }

# ---------------- TRAIN ----------------

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_macro_f1 = -1.0
history = []

for epoch in range(EPOCHS):
    model.train()
    epoch_losses = []
    start_time = time.time()

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for batch in progress:
        batch = {
            key: value.to(device, non_blocking=True)
            for key, value in batch.items()
        }

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=True):
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        epoch_losses.append(loss.item())
        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            lr=f"{scheduler.get_last_lr()[0]:.2e}"
        )

    val_metrics = evaluate(val_loader)

    epoch_result = {
        "epoch": epoch + 1,
        "train_loss": float(np.mean(epoch_losses)),
        "validation_loss": val_metrics["loss"],
        "validation_accuracy": val_metrics["accuracy"],
        "validation_macro_f1": val_metrics["macro_f1"],
        "seconds": round(time.time() - start_time, 1)
    }
    history.append(epoch_result)

    print("\nEpoch result:")
    print(json.dumps(epoch_result, indent=2))

    if val_metrics["macro_f1"] > best_macro_f1:
        best_macro_f1 = val_metrics["macro_f1"]

        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)

        with open(OUTPUT_DIR / "label_mapping.json", "w") as f:
            json.dump(
                {"label2id": LABEL2ID, "id2label": ID2LABEL},
                f,
                indent=2
            )

        print("Saved new best checkpoint:", OUTPUT_DIR)

# Save training history.
with open(OUTPUT_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)

# ---------------- FINAL TEST EVALUATION ----------------
# Test data is evaluated only after selecting the best checkpoint by validation macro-F1.

del model
torch.cuda.empty_cache()

model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
model.to(device)

test_metrics = evaluate(test_loader)

print("\nFINAL TEST RESULTS")
print("Accuracy:", round(test_metrics["accuracy"], 4))
print("Macro F1:", round(test_metrics["macro_f1"], 4))
print("Test loss:", round(test_metrics["loss"], 4))

report = classification_report(
    test_metrics["targets"],
    test_metrics["predictions"],
    labels=list(range(len(LABELS))),
    target_names=LABELS,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).transpose()
display(report_df)

report_df.to_csv(OUTPUT_DIR / "test_classification_report.csv")

cm = confusion_matrix(
    test_metrics["targets"],
    test_metrics["predictions"],
    labels=list(range(len(LABELS)))
)

pd.DataFrame(
    cm,
    index=LABELS,
    columns=LABELS
).to_csv(OUTPUT_DIR / "test_confusion_matrix.csv")

with open(OUTPUT_DIR / "test_metrics.json", "w") as f:
    json.dump({
        "checkpoint": CHECKPOINT,
        "architecture": "BERT-Tiny E2",
        "labels": LABELS,
        "accuracy": test_metrics["accuracy"],
        "macro_f1": test_metrics["macro_f1"],
        "test_loss": test_metrics["loss"],
        "embedding_transfer_coverage": matched / len(new_vocab),
        "history": history
    }, f, indent=2)

# Zip final artifacts for download.
import shutil
zip_path = shutil.make_archive(
    "/content/bert_tiny_e2_multilingual4.4",
    "zip",
    OUTPUT_DIR
)

print("\nModel artifacts:", zip_path)
print("Best validation macro F1:", round(best_macro_f1, 4))

from google.colab import files
files.download(zip_path)

GPU: Tesla T4
PyTorch: 2.11.0+cu130


model.safetensors: reconstructing file:   0%|          |  0.00B / 17.7MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...


FINAL TEST RESULTS
Accuracy: 0.9971
Macro F1: 0.9971
Test loss: 0.0137


,precision,recall,f1-score,support
as-IN,0.995529,0.996840,0.996184,7595.000000
bn-IN,0.997483,0.996105,0.996793,7958.000000
en-IN,0.997916,0.994162,0.996036,7708.000000
gu-IN,0.996600,0.996731,0.996666,7647.000000
hi-IN,0.992373,0.995591,0.993979,7711.000000
kn-IN,0.998256,0.997185,0.997720,7460.000000
ml-IN,0.999195,0.998524,0.998859,7454.000000
mr-IN,0.996460,0.997244,0.996852,7621.000000
pa-IN,0.996953,0.998143,0.997548,7540.000000
ta-IN,0.998921,0.999055,0.998988,7411.000000



Model artifacts: /content/bert_tiny_e2_multilingual4.4.zip
Best validation macro F1: 0.9973


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

TESTING MY MODEL

In [ ]:
import zipfile

with zipfile.ZipFile('/content/opensourceModel_langaueclassifier_V!.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/model')


In [ ]:
import pandas as pd

In [ ]:
eval_df=pd.read_csv("/content/language_id_eval_80k.csv")

In [ ]:
eval_df

,eval_id,text,label,label_id,category,script,source_type,language_detail,variant,source_dataset,hard_negative,n_chars,n_words
0,eval_000000,পর্যটকরা যেতে পারে বিভিন্ন নিদর্শনে কোন নির্দি...,bn-IN,1,travel,Bengali,real,ben_Beng,clean_sentence,flores200_sib200,False,118,17
1,eval_000001,Shabdagalannu beresi prakriyegolapadisuva salu...,kn-IN,5,science/technology,Latin,derived_transform,kan_Knda,romanized_synthetic,flores200_sib200,False,136,11
2,eval_000002,Ala gor ane janaral tomi phrenks samany rite t...,gu-IN,3,entertainment,Latin,derived_transform,guj_Gujr,romanized_synthetic,flores200_sib200,False,239,37
3,eval_000003,ਵਿਗਿਆਨੀਆਂ ਦਾ ਕਹਿਣਾ ਹੈ ਕਿ ਟਕਰਾਅ ਕਾਰਨ ਹੋਇਆ ਧਮਾਕਾ...,pa-IN,8,science/technology,Gurmukhi,derived_transform,pan_Guru,misc_wrapped,flores200_sib200,False,65,14
4,eval_000004,nyukliyas do kanom nyutron aur proton se milak...,hi-IN,4,science/technology,Latin,derived_transform,hin_Deva,romanized_synthetic,flores200_sib200,False,58,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...
87469,eval_087469,"praktana haucha spikaro niuta jinajaricho, tek...",as-IN,0,politics,Latin,derived_transform,asm_Beng,romanized_synthetic,flores200_sib200,False,172,20
87470,eval_087470,ਸੰਗੀਤ ਆਮ ਕਰਕੇ ਅਵਾਜ਼ਾਂ ਨੂੰ ਇਕੱਠਿਆ ਤਰਕੀਬ ਦੇਣ ਅਤੇ ...,pa-IN,8,science/technology,Gurmukhi,real,pan_Guru,clean_sentence,flores200_sib200,False,107,21
87471,eval_087471,"Kiam vi telefonas al iu, kiu estas milojn da m...",none,11,science/technology,Latin,real,epo_Latn,clean_sentence,flores200_sib200,False,75,14
87472,eval_087472,এই টেলিস্ক'পটোৱে দূৰৱৰ্তী বস্তু এটা ওচৰত আৰু ড...,as-IN,0,science/technology,Bengali,real,asm_Beng,clean_sentence,flores200_sib200,False,99,15


In [1]:
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

# 1. Load the evaluation data
# csv_path = "eval_df.csv"  # Replace with your actual CSV path if different
# df = pd.read_csv(csv_path)

# 2. Define the path where you extracted the model
model_path = "/content/model"

# 3. Load the tokenizer and model from the local directory
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

# 4. Create a text classification pipeline
classifier = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)
# Note: Remove device=0 if you are not using a GPU

# 5. Extract text from the appropriate column and predict
# Replace 'text_column_name' with the actual name of the column containing your text
text_data = eval_df['text'].tolist()

# Run predictions
predictions = classifier(text_data)

# 6. Extract the predicted labels and add to the DataFrame
eval_df['predicted'] = [pred['label'] for pred in predictions]

# 7. Save or view the updated DataFrame
# eval_df.to_csv("evaluated_eval_df.csv", index=False)
# print("Evaluation complete! Saved to 'evaluated_eval_df.csv'.")
